# ASPER — Task 3: Random Forest Machine Learning Model

## Titanic Survival Prediction

**Objective:** Build a Random Forest classification model that predicts whether a Titanic passenger survived using passenger information.

Workflow: data loading → cleaning → feature engineering → EDA → preprocessing → training/testing → evaluation.


## 1. Problem Statement

Develop a Machine Learning model using the Random Forest algorithm to predict Titanic passenger survival.

- `survived = 0`: did not survive
- `survived = 1`: survived

**Dataset source:** https://github.com/mwaskom/seaborn-data/blob/master/titanic.csv


In [ ]:
# Import libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    classification_report, confusion_matrix, ConfusionMatrixDisplay
)


## 2. Load the Dataset

The default dataset is loaded from the public Seaborn dataset repository. An optional Google Colab upload cell is provided below.


In [ ]:
url = "https://raw.githubusercontent.com/mwaskom/seaborn-data/master/titanic.csv"
df = pd.read_csv(url)

print("Dataset shape:", df.shape)
display(df.head())


In [ ]:
# OPTIONAL — Google Colab CSV upload
# Run this cell only if you want to replace the default dataset.

from google.colab import files

uploaded = files.upload()
if uploaded:
    uploaded_name = next(iter(uploaded))
    df = pd.read_csv(uploaded_name)
    print("Uploaded:", uploaded_name)
    print("Dataset shape:", df.shape)
    display(df.head())


## 3. Data Inspection

We inspect columns, data types, missing values, and duplicate records before training.


In [ ]:
print("Columns:")
print(df.columns.tolist())

print("\nData types:")
print(df.dtypes)

print("\nMissing values:")
display(df.isnull().sum().sort_values(ascending=False).to_frame("missing_values"))

print("Duplicate rows:", df.duplicated().sum())


## 4. Data Cleaning

Duplicate records are removed. Missing numerical values will be handled using median imputation, while missing categorical values will be handled using the most frequent category.


In [ ]:
before = len(df)
df = df.drop_duplicates().reset_index(drop=True)
print(f"Removed {before - len(df)} duplicate rows.")

df = df.dropna(subset=["survived"]).reset_index(drop=True)


## 5. Feature Engineering

A `family_size` feature is created from `sibsp` and `parch`.

`family_size = sibsp + parch + 1`

We then separate the input features `X` from the target `y`.


In [ ]:
df["family_size"] = df["sibsp"] + df["parch"] + 1

features = [
    "pclass", "sex", "age", "sibsp", "parch", "fare",
    "embarked", "class", "who", "alone", "family_size"
]

X = df[features].copy()
y = df["survived"].astype(int)

print("Features:")
print(features)
print("\nTarget: survived")


## 6. Exploratory Data Analysis (EDA)

EDA means exploring the dataset before modeling. We examine distributions and relationships that may help us understand the data.


In [ ]:
plt.figure(figsize=(6,4))
sns.countplot(data=df, x="survived")
plt.title("Survival Distribution")
plt.xlabel("Survived (0 = No, 1 = Yes)")
plt.ylabel("Number of passengers")
plt.show()

plt.figure(figsize=(6,4))
sns.countplot(data=df, x="pclass", hue="survived")
plt.title("Survival by Passenger Class")
plt.xlabel("Passenger class")
plt.ylabel("Number of passengers")
plt.show()

plt.figure(figsize=(7,4))
sns.histplot(data=df, x="age", hue="survived", kde=True, bins=25)
plt.title("Age Distribution by Survival")
plt.show()


### EDA observations

- The target contains two classes: survived and did not survive.
- Passenger class and sex provide useful information related to survival.
- Age contains missing values, which are handled during preprocessing.
- EDA helps us understand the dataset before model training.


## 7. Train/Test Split

The dataset is divided into training and testing sets. The model learns from the training set and is evaluated on unseen test data.


In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y
)

print("Training samples:", len(X_train))
print("Testing samples:", len(X_test))


## 8. Preprocessing

- Numerical missing values → median imputation.
- Categorical missing values → most-frequent imputation.
- Categorical variables → one-hot encoding.


In [ ]:
numeric_features = ["pclass", "age", "sibsp", "parch", "fare", "family_size"]
categorical_features = ["sex", "embarked", "class", "who", "alone"]

numeric_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median"))
])

categorical_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore"))
])

preprocessor = ColumnTransformer(transformers=[
    ("num", numeric_transformer, numeric_features),
    ("cat", categorical_transformer, categorical_features)
])


## 9. Random Forest Model

Random Forest combines multiple decision trees. For classification, the trees vote on the final predicted class.


In [ ]:
model = RandomForestClassifier(
    n_estimators=200,
    random_state=42,
    class_weight="balanced"
)

pipeline = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("model", model)
])

pipeline.fit(X_train, y_train)
print("Random Forest model trained successfully.")


## 10. Evaluation

We calculate accuracy, precision, recall, F1-score, a classification report, and a confusion matrix.


In [ ]:
y_pred = pipeline.predict(X_test)

accuracy = accuracy_score(y_test, y_pred)
precision = precision_score(y_test, y_pred, zero_division=0)
recall = recall_score(y_test, y_pred, zero_division=0)
f1 = f1_score(y_test, y_pred, zero_division=0)

results = pd.DataFrame({
    "Metric": ["Accuracy", "Precision", "Recall", "F1-score"],
    "Score": [accuracy, precision, recall, f1]
})

display(results)

print("\nClassification Report:")
print(classification_report(
    y_test, y_pred,
    target_names=["Did not survive", "Survived"],
    zero_division=0
))


In [ ]:
cm = confusion_matrix(y_test, y_pred)
disp = ConfusionMatrixDisplay(
    confusion_matrix=cm,
    display_labels=["Did not survive", "Survived"]
)
disp.plot()
plt.title("Random Forest Confusion Matrix")
plt.show()


## 11. Results and Observations

Run the notebook from top to bottom in Google Colab. The evaluation cell calculates the actual metrics from the held-out test set.

**Do not manually invent metric values.** Use the values produced by the executed notebook in the final submission.

The confusion matrix shows the model's correct and incorrect predictions for both classes.


## 12. Conclusion

A Random Forest classification model was developed for Titanic survival prediction. The project includes data inspection, duplicate removal, missing-value handling, feature engineering, EDA, categorical encoding, train/test splitting, model training, and evaluation.

The final accuracy, precision, recall, and F1-score should be reported from the executed notebook.


## 13. Viva / Recruitment Quick Questions

**What is Random Forest?**  
An ensemble algorithm that combines multiple decision trees. For classification, the final class is generally based on the trees' votes.

**What are features?**  
Input variables used to make predictions.

**What is the target?**  
The output variable being predicted. Here it is `survived`.

**Why train/test split?**  
To train on one portion of the data and evaluate on unseen data.

**What is EDA?**  
Exploratory Data Analysis: examining distributions, relationships, missing values, duplicates, and other properties before modeling.

**Why handle missing values?**  
Missing values need to be handled appropriately so preprocessing and model training can work reliably.

**What is one-hot encoding?**  
Converting categorical values into numerical indicator columns.

**What is accuracy?**  
The proportion of predictions that are correct.

**What is a confusion matrix?**  
A table showing correct and incorrect predictions for each class.
